# `nisar_access_subset` — DPS job runner

Find a NISAR GCOV granule over an area of interest and submit a MAAP DPS job to the
`nisar_access_subset` process, which subsets the requested variables (optionally by a
bounding box) and writes a Zarr store. Run this in a MAAP Hub workspace with an Earthdata
Login.

AOI: Washington, DC — `POLYGON((-77.1 38.8,-76.8 38.8,-76.8 39,-77.1 39,-77.1 38.8))`.

In [ ]:
# %pip install earthaccess   # one-time, if needed
import earthaccess
from maap.maap import MAAP

maap = MAAP()
earthaccess.login()  # NISAR collections are auth-gated; required for search.

## 1. Find granules (earthaccess / CMR)

In [ ]:
BBOX = (-77.1, 38.8, -76.8, 39.0)  # (W, S, E, N) — Washington, DC

results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=BBOX,
    temporal=("2025-10-01", "2025-10-31"),
)
print(f"{len(results)} granules")

def first_h5(links):
    return next((u for u in (links or []) if u.lower().endswith(".h5")), None)

# Extract each granule's https and s3 .h5 links: these are the algorithm's inputs. The
# algorithm is written in a cloud-optimized way and reads the granule directly from these
# links (S3-preferred, https fallback) — we do not download the data here.
granules = [
    {
        "name": g["umm"]["GranuleUR"],
        "s3": first_h5(g.data_links(access="direct")),
        "https": first_h5(g.data_links()),
    }
    for g in results
]
for gr in granules:
    print(gr["name"])

## 2. Resolve the deployed process id

`submit_job` needs the numeric `processID`, not the string `id`.

In [ ]:
procs = maap.list_algorithms().json().get("processes", [])
matches = [p for p in procs if "nisar_access_subset" in str(p.get("id", "")).lower()]
PROCESS_ID = max((p["processID"] for p in matches), default=None)
print("PROCESS_ID =", PROCESS_ID)

## 3. Submit a job per granule

Submit one DPS job per search result (the algorithm processes a single granule per job). The
worker downloads each granule itself (S3-preferred, HTTPS fallback). The subset reuses the AOI
`BBOX` from above — CMR's `(W, S, E, N)` is the same `minx,miny,maxx,maxy` order the `bbox`
input expects. Set `bbox=""` to keep the full granule extent.

In [ ]:
bbox_str = ",".join(str(c) for c in BBOX)  # reuse the AOI bbox: minx,miny,maxx,maxy

# Safety limit while testing: only submit the first MAX_JOBS granules so a wide AOI/time
# window doesn't fire off many jobs at once. Set to len(granules) to submit all.
MAX_JOBS = 1

job_ids = []
for gr in granules[:MAX_JOBS]:
    inputs = {
        "access_mode": "auto",
        "s3_href": gr["s3"],
        "https_href": gr["https"],
        "vars": "HHHH",
        "group": "/science/LSAR/GCOV/grids/frequencyA",
        "bbox": bbox_str,
        "bbox_crs": "EPSG:4326",
        "out_name": "nisar_subset.zarr",
    }
    r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,
                        queue="maap-dps-worker-8gb", tag="nisar-access-subset")
    body = r.json()
    job_id = body.get("jobID") or body.get("id")
    job_ids.append(job_id)
    print(r.status_code, job_id, gr["name"])

## 4. Monitor and fetch results

In [ ]:
for job_id in job_ids:
    print(job_id, "->", maap.get_job_status(job_id).json().get("status"))
# Once a job succeeds, inspect its outputs (Zarr store + manifest.json):
# print(maap.get_job_result(job_ids[0]).json())

## 5. Locate outputs and plot

`get_job_result` returns the job's output URL(s). DPS stages those outputs to your MAAP
private bucket, which is mounted locally at `~/my-private-bucket/` — the `dps_output/...`
tail of the URL is the path under that mount. Below we derive the exact output folder from
the result (no filesystem search), then open the Zarr store (`out_name` from §3) and plot the
first variable. (Needs `xarray`, `zarr`, and `matplotlib`.)

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

OUT_NAME = "nisar_subset.zarr"  # must match out_name submitted in §3
PRIVATE_BUCKET = os.path.expanduser("~/my-private-bucket")

# Ask DPS for the job's result. It returns {prop: {"links": [{"href": ...}, ...]}}; collect
# every href and keep the first that carries the "dps_output/" path.
job_id = job_ids[0]
result = maap.get_job_result(job_id).json()
hrefs = [l["href"] for p in result.values() for l in p.get("links", []) if "href" in l]
output_url = next(h for h in hrefs if "dps_output/" in h)
print("Job output URL:", output_url)

# The URL's "dps_output/..." tail maps directly onto the local private-bucket mount.
tail = output_url.split("dps_output/", 1)[1].rstrip("/")
out_dir = os.path.join(PRIVATE_BUCKET, "dps_output", tail)
print("Output location:", out_dir)
print("Contents:", sorted(os.listdir(out_dir)))

# Open the known Zarr store (no searching) and plot the first variable.
zarr_path = os.path.join(out_dir, OUT_NAME)
ds = xr.open_zarr(zarr_path, consolidated=True)
var = list(ds.data_vars)[0]
print("Plotting variable:", var, "shape:", ds[var].shape)

data = np.abs(ds[var].values)  # GCOV samples may be complex; show magnitude
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(data, origin="upper", cmap="gray")
ax.set_title(f"{var} — {OUT_NAME}")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()